# Ridership exploration — UPT, VRM, VRH, with coverage context

> Explore and compute freely: nothing computed outside Headway's calculation library (services/calc) can ever become a reported figure. Only the calculation library writes computed.metric_values, and the walls are structural database CHECKs, not policy.

This notebook uses [`headway-client`](../clients/python/README.md) to pull the
ridership and service figures Headway computed — unlinked passenger trips
(UPT), vehicle revenue miles (VRM), vehicle revenue hours (VRH) — and shows
what makes a Headway figure different: every row arrives with its full
provenance (which calculation version produced it, whether it is certified,
whether simulated data is anywhere underneath it), the coverage/completeness
accounting the calculation persisted about its own run, and a lineage trail
you can walk to the raw records.

**You need:** `HEADWAY_API_URL` (e.g. `http://127.0.0.1:8000`) and
`HEADWAY_MACHINE_KEY` — a machine API key with the `read:metrics`
permission; an administrator issues one per
[docs/analyst-access.md](../docs/analyst-access.md).

*This notebook was executed once against a live Headway demo stack (2026-07-15) so its outputs are real; some demo figures come from simulated feeds and say so in their provenance. To run it yourself you need a running Headway API and the environment variables named below — credentials are never written into a notebook.*

In [1]:
import os

import pandas as pd

from headway_client import HeadwayClient, frames

pd.set_option("display.width", 160)

hw = HeadwayClient(
    os.environ.get("HEADWAY_API_URL", "http://127.0.0.1:8000"),
    token=os.environ["HEADWAY_MACHINE_KEY"],  # never paste a key into a notebook
)

## The figures, with provenance always attached

`frames.metric_values_frame` always includes the provenance columns —
`metric_value_id`, `calc_name`, `calc_version`, `category`,
`certification_status`, `simulated`, `source_mix`. There is no option to
omit them: narrowing what you *print* (as below, dropping the verbose
`detail` column from the view) is your explicit act, and the frame itself
keeps everything.

In [2]:
ridership = [
    v
    for metric in ("upt", "vrm", "vrh")
    for v in hw.metric_values(metric=metric, category="ntd")
]
df = frames.metric_values_frame(ridership)
print(f"{len(df)} figures; columns: {list(df.columns)}")
df.drop(columns=["detail", "source_mix", "computed_at"]).sort_values(
    ["metric", "period_start", "scope"]
).head(12)

54 figures; columns: ['metric', 'period_start', 'period_end', 'scope', 'value', 'unit', 'metric_value_id', 'calc_name', 'calc_version', 'category', 'certification_status', 'simulated', 'source_mix', 'computed_at', 'detail']


,metric,period_start,period_end,scope,value,unit,metric_value_id,calc_name,calc_version,category,certification_status,simulated
0,upt,2026-07-09,2026-07-10,agency,238100,unlinked_passenger_trips,bd22d723-ac7e-43ce-acad-0d9191fcc1f0,upt_v0,0.1.0,ntd,uncertified,True
1,upt,2026-07-09,2026-07-10,agency,238100,unlinked_passenger_trips,06403f2f-dc2e-4aaf-afde-5572033a0a17,upt_v0,0.1.0,ntd,uncertified,True
2,upt,2026-07-09,2026-07-10,agency,238100,unlinked_passenger_trips,538e8e79-99cb-4639-afc2-2b11f356b017,upt_v0,0.1.0,ntd,uncertified,True
3,upt,2026-07-09,2026-07-10,mode:bus,173887,unlinked_passenger_trips,bc190603-201a-4df0-9310-ee4f7711bcdc,upt_v0,0.1.0,ntd,uncertified,True
4,upt,2026-07-09,2026-07-10,mode:rail,5489,unlinked_passenger_trips,8b0e13dc-f094-49c4-91f5-0a2b33715872,upt_v0,0.1.0,ntd,uncertified,True
5,upt,2026-07-09,2026-07-10,mode:subway,7011,unlinked_passenger_trips,da3cf0a6-a3af-42c5-bad9-88f6bdac410b,upt_v0,0.1.0,ntd,uncertified,True
6,upt,2026-07-09,2026-07-10,mode:tram,17508,unlinked_passenger_trips,d1fecac1-29d0-4bed-86b3-9972534b4788,upt_v0,0.1.0,ntd,uncertified,True
7,upt,2026-07-09,2026-07-10,mode:unknown,34204,unlinked_passenger_trips,61abe48e-8722-44f1-a655-3667a0516941,upt_v0,0.1.0,ntd,uncertified,True
8,upt,2026-07-14,2026-07-16,agency,0,unlinked_passenger_trips,ac19a418-2099-4245-a9d4-42d28cad6011,upt_v0,0.1.0,ntd,uncertified,False
13,upt,2026-07-14,2026-07-16,agency,0,unlinked_passenger_trips,a6c8727c-ff1c-4488-8865-45438bfa157d,upt_v0,0.1.0,ntd,uncertified,False


Figures are exact `decimal.Decimal` objects parsed from the API's
decimal strings — floating point never touches a reported value, so sums
like this are exact:

In [3]:
agency = df[df["scope"] == "agency"]
agency.groupby("metric")["value"].agg(["count", "min", "max"])

,count,min,max
metric,,,
upt,5,0,238100
vrh,7,0.00,16326.89
vrm,7,0.00,160835.49


## Coverage context — the calculation's own honesty accounting

A VRM/VRH figure carries the coverage detail its calculation persisted:
how many vehicle-day groups it saw, how many it *excluded* (each exclusion
is an owned data-quality issue, never a silent drop), and the coverage
threshold the run was held to. A run that cannot meet its bar **refuses to
emit a certifiable figure** rather than interpolating — so a figure you can
read here is a figure that passed its own stated test.

In [4]:
coverage_rows = df[df["detail"].apply(lambda d: "coverage" in d)]
coverage = pd.DataFrame(
    {
        "metric": coverage_rows["metric"],
        "scope": coverage_rows["scope"],
        "period_start": coverage_rows["period_start"],
        "value": coverage_rows["value"],
        "certification_status": coverage_rows["certification_status"],
        # detail ratios are strings on purpose (exactness); shown verbatim
        "coverage": coverage_rows["detail"].apply(lambda d: d["coverage"]),
        "coverage_threshold": coverage_rows["detail"].apply(
            lambda d: d.get("coverage_threshold")
        ),
        "total_groups": coverage_rows["detail"].apply(
            lambda d: d.get("total_groups")
        ),
        "excluded_groups": coverage_rows["detail"].apply(
            lambda d: d.get("excluded_groups")
        ),
    }
)
coverage.sort_values(["metric", "period_start", "scope"]).head(10)

,metric,scope,period_start,value,certification_status,coverage,coverage_threshold,total_groups,excluded_groups
37,vrh,agency,2026-07-01,16326.89,uncertified,0.9061,0.90,9845,1205
38,vrh,mode:bus,2026-07-01,13448.98,uncertified,0.9498,0.90,3593,12
36,vrh,mode:rail,2026-07-01,690.72,uncertified,0.9143,0.90,712,61
39,vrh,agency,2026-07-09,1260.85,certified,0.9263,0.90,2742,202
41,vrh,agency,2026-07-09,5364.54,uncertified,0.9146,0.90,3340,362
42,vrh,agency,2026-07-09,5364.54,uncertified,0.9146,0.90,3340,362
40,vrh,mode:rail,2026-07-09,214.24,uncertified,0.9573,0.95,211,9
43,vrh,agency,2026-07-10,5389.40,uncertified,0.9116,0.90,4027,473
47,vrh,agency,2026-07-14,0.00,uncertified,1.0000,0.95,0,0
52,vrh,agency,2026-07-14,0.00,uncertified,1.0000,0.95,0,0


## UPT's missing-trip accounting, and the simulated-data rule

UPT details carry the missing-trip arithmetic (operated trips vs trips
with passenger events, the share, and the factor applied when the share is
under the FTA line) plus `source_mix` — record counts per envelope source.
Any source containing `simulated` marks the whole figure `simulated=True`,
permanently: simulated data can inform exploration, never a submission.

In [5]:
upt = df[df["metric"] == "upt"].copy()
upt["missing_share"] = upt["detail"].apply(lambda d: d.get("missing_share"))
upt["factor_applied"] = upt["detail"].apply(lambda d: d.get("factor_applied"))
upt[["scope", "period_start", "value", "missing_share", "factor_applied",
     "simulated", "source_mix", "certification_status"]].head(10)

,scope,period_start,value,missing_share,factor_applied,simulated,source_mix,certification_status
0,agency,2026-07-09,238100,0.0100,1.010075,True,{'tides_simulated': 111568},uncertified
1,agency,2026-07-09,238100,0.0100,1.010075,True,{'tides_simulated': 111568},uncertified
2,agency,2026-07-09,238100,0.0100,1.010075,True,{'tides_simulated': 111568},uncertified
3,mode:bus,2026-07-09,173887,0.0105,1.010630,True,{'tides_simulated': 81376},uncertified
4,mode:rail,2026-07-09,5489,0.0048,1.004854,True,{'tides_simulated': 2511},uncertified
5,mode:subway,2026-07-09,7011,0.0000,1.000000,True,{'tides_simulated': 3339},uncertified
6,mode:tram,2026-07-09,17508,0.0132,1.013373,True,{'tides_simulated': 8271},uncertified
7,mode:unknown,2026-07-09,34204,0.0084,1.008455,True,{'tides_simulated': 16071},uncertified
8,agency,2026-07-14,0,0.0000,1.000000,False,{},uncertified
9,mode:DR,2026-07-14,204,NaN,NaN,True,{'dr_simulated': 95},uncertified


## Walk one lineage trail — the figure proves itself

Every `metric_value_id` above is an input to `walk_lineage`, which returns
the full "explain this number" trail down to the immutable,
content-addressed raw records the figure was computed from.

In [6]:
certified = df[df["certification_status"] == "certified"]
target = (certified if len(certified) else df).iloc[0]
trail = hw.walk_lineage(target["metric_value_id"])
lineage = frames.lineage_frame(trail)
print(
    f"{target['metric']} = {target['value']} {target['unit']} "
    f"({target['certification_status']}, {target['calc_name']} "
    f"{target['calc_version']})\n"
    f"trail: {len(lineage)} nodes, bottoming out at "
    f"{len(trail.raw_record_ids())} raw records"
)
lineage.head(6)

vrm = 12794.92 miles (certified, vrm_v0 0.2.0)
trail: 327 nodes, bottoming out at 326 raw records


,depth,kind,id,transform_name,transform_version,parent_kind,parent_id
0,0,computed.metric_values,b3ebdef6-ba9e-44fa-97af-7f487bc2aca5,vrm_v0,0.2.0,NaN,NaN
1,1,raw.records,002da97614ea64e3f2253ded0bf1d10869c5c54f8b9cc6...,NaN,NaN,computed.metric_values,b3ebdef6-ba9e-44fa-97af-7f487bc2aca5
2,1,raw.records,00f658bb4de1889d5fa5bebb8ec9d20f20b3947b25b8cf...,NaN,NaN,computed.metric_values,b3ebdef6-ba9e-44fa-97af-7f487bc2aca5
3,1,raw.records,029fb722c858a169be379c4d0760de1ebec0a2a30544ef...,NaN,NaN,computed.metric_values,b3ebdef6-ba9e-44fa-97af-7f487bc2aca5
4,1,raw.records,02a04789d576a84cb09274450bebfa1bab0e91d9f8e4bd...,NaN,NaN,computed.metric_values,b3ebdef6-ba9e-44fa-97af-7f487bc2aca5
5,1,raw.records,02b4ddc2285cbddd59fae27f43a9398e4539a2196e736e...,NaN,NaN,computed.metric_values,b3ebdef6-ba9e-44fa-97af-7f487bc2aca5


Those `raw.records` ids are SHA-256 digests of the raw payload
bytes as received — the bottom of the provenance chain. If a figure ever
needed defending in an FTA review, this walk is the defense.

**Next:** operations metrics with honest labeling
([02-otp-headway-adherence](02-otp-headway-adherence.ipynb)), data-quality
triage ([03-dq-triage](03-dq-triage.ipynb)), and the access setup guide
([docs/analyst-access.md](../docs/analyst-access.md)).